In [12]:
!python -V

Python 3.9.16


# Homework Experiment Tracking 

Note: Create and initiate the corresponding environment `exp-tracking-env` prior to doing the task. This will install all the packages required to run experiments and register models in MLFlow. 

To activate the environment

    source /Users/Personas/miniconda3/bin/activate

    conda activate exp-tracking-env`

To setup the environment and install the requirements

    conda create -n exp-tracking-env python=3.9

    pip install -r requirements.txt

To initiate `mlflow` type `mlflow` in the terminal



## Q1. Install the package
---

To get started with MLflow you'll need to install the appropriate Python package.

For this we recommend creating a separate Python environment, for example, you can use [conda environments](https://docs.conda.io/projects/conda/en/latest/user-guide/getting-started.html#managing-envs), 
and then install the package there with `pip` or `conda`.

Once you installed the package, run the command `mlflow --version` and check the output.

What's the version that you have?

**Answer**

 --> 2.4.1 <--

## Q2. Download and preprocess the data
---

We'll use the Green Taxi Trip Records dataset to predict the amount of tips for each trip. 

Download the data for January, February and March 2022 in parquet format from [here](https://www1.nyc.gov/site/tlc/about/tlc-trip-record-data.page).

Use the script `preprocess_data.py` located in the folder [`homework`](homework) to preprocess the data.

The script will:

* load the data from the folder `<TAXI_DATA_FOLDER>` (the folder where you have downloaded the data),
* fit a `DictVectorizer` on the training set (January 2022 data),
* save the preprocessed datasets and the `DictVectorizer` to disk.

Your task is to download the datasets and then execute this command:

`python preprocess_data.py --raw_data_path <TAXI_DATA_FOLDER> --dest_path ./output`


In [2]:
import requests
import os


def download_data(url, filename, folder_path):
# Create the data folder if it doesn't exist
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

    file_path = os.path.join(folder_path, filename)

    response = requests.get(url)
    if response.status_code == 200:
        with open(file_path, 'wb') as file:
            file.write(response.content)
        print(f"File {filename} downloaded and saved successfully.")
    else:
        print(f"Failed to download the file {filename}")
    

In [3]:
download_data('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2022-01.parquet',
              "green_tripdata_2022-01.parquet", 
              'data')

download_data('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2022-02.parquet',
              'green_tripdata_2022-02.parquet',
              'data')

download_data('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2022-03.parquet',
              'green_tripdata_2022-03.parquet',
              'data')


File green_tripdata_2022-01.parquet downloaded and saved successfully.
File green_tripdata_2022-02.parquet downloaded and saved successfully.
File green_tripdata_2022-03.parquet downloaded and saved successfully.


In [ ]:

# Execute the command

import subprocess

command = "python preprocess_data.py --raw_data_path ./data --dest_path ./output"
result = subprocess.run(command, shell=True, capture_output=True, text=True)

# Print the output and error, if any
print("Output:")
print(result.stdout)

if result.stderr:
    print("Error:")
    print(result.stderr)

In [10]:
filepath = './output/dv.pkl'
dv_bytes= os.path.getsize(filepath) # size in bytes

dv_kb = dv_bytes/1024
dv_kb


150.05859375

**Answer**

--> 154kB <--

## Q3. Train a model with autolog
---

We will train a `RandomForestRegressor` (from Scikit-Learn) on the taxi dataset.

We have prepared the training script `train.py` for this exercise, which can be also found in the folder `homework`. 

The script will:

* load the datasets produced by the previous step,
* train the model on the training set,
* calculate the RMSE score on the validation set.

Your task is to modify the script to enable **autologging** with MLflow, execute the script and then launch the MLflow UI to check that the experiment run was properly tracked. 

Tip 1: don't forget to wrap the training code with a `with mlflow.start_run():` statement as we showed in the videos.

Tip 2: don't modify the hyperparameters of the model to make sure that the training will finish quickly.

What is the value of the `max_depth` parameter:

* 4
* 6
* 8
* --> 10 <--

In [13]:
# Execute the train.py 
# Add in the code
#    mlflow.set_tracking_uri('sqlite:///mlflow.db')
#    mlflow.set_experiment('rf_train')
#    mlflow.autolog()

!python3 train.py


2023/06/13 10:48:22 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2023/06/13 10:48:22 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running upgrade  -> 451aebb31d03, add metric step
INFO  [alembic.runtime.migration] Running upgrade 451aebb31d03 -> 90e64c465722, migrate user column to tags
INFO  [alembic.runtime.migration] Running upgrade 90e64c465722 -> 181f10493468, allow nulls for metric values
INFO  [alembic.runtime.migration] Running upgrade 181f10493468 -> df50e92ffc5e, Add Experiment Tags Table
INFO  [alembic.runtime.migration] Running upgrade df50e92ffc5e -> 7ac759974ad8, Update run tags with larger limit
INFO  [alembic.runtime.migration] Running upgrade 7ac759974ad8 -> 89d4b8295536, create latest metrics table
INFO  [89d4b8295536_create_latest_metrics_table_py] Migration complete!
INFO  

## Launch the tracking server locally for MLflow

Now we want to manage the entire lifecycle of our ML model. In this step, you'll need to launch a tracking server. This way we will also have access to the model registry. 

In case of MLflow, you need to:

* launch the tracking server on your local machine,
* select a SQLite db for the backend store and a folder called `artifacts` for the artifacts store.

You should keep the tracking server running to work on the next three exercises that use the server.

In [ ]:
mlflow ui --backend-store-uri sqlite:///mlflow.db -p 5001

# Q4. Tune model hyperparameters
---

Now let's try to reduce the validation error by tuning the hyperparameters of the RandomForestRegressor using optuna. We have prepared the script hpo.py for this exercise.

Your task is to modify the script hpo.py and make sure that the validation RMSE is logged to the tracking server for each run of the hyperparameter optimization (you will need to add a few lines of code to the objective function) and run the script without passing any parameters.

After that, open UI and explore the runs from the experiment called random-forest-hyperopt to answer the question below.

Note: Don't use autologging for this exercise.

The idea is to just log the information that you need to answer the question below, including:

the list of hyperparameters that are passed to the objective function during the optimization,
the RMSE obtained on the validation set (February 2022 data).
What's the best validation RMSE that you got?

- 1.85
- 2.15
- --> 2.45 <---
- 2.85

In [15]:
!python3 hpo.py

2023/06/13 11:05:54 INFO mlflow.tracking.fluent: Experiment with name 'random-forest-hyperopt' does not exist. Creating a new experiment.
[I 2023-06-13 11:05:54,808] A new study created in memory with name: no-name-f8376c23-8f85-4226-a993-a0ed494c601b
[I 2023-06-13 11:05:55,599] Trial 0 finished with value: 2.451379690825458 and parameters: {'n_estimators': 25, 'max_depth': 20, 'min_samples_split': 8, 'min_samples_leaf': 3}. Best is trial 0 with value: 2.451379690825458.
[I 2023-06-13 11:05:55,651] Trial 1 finished with value: 2.4667366020368333 and parameters: {'n_estimators': 16, 'max_depth': 4, 'min_samples_split': 2, 'min_samples_leaf': 4}. Best is trial 0 with value: 2.451379690825458.
[I 2023-06-13 11:05:55,960] Trial 2 finished with value: 2.449827329704216 and parameters: {'n_estimators': 34, 'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 4}. Best is trial 2 with value: 2.449827329704216.
[I 2023-06-13 11:05:56,089] Trial 3 finished with value: 2.460983516558473 a

# Q5. Promote the best model to the model registry
----

The results from the hyperparameter optimization are quite good. So, we can assume that we are ready to test some of these models in production. In this exercise, you'll promote the best model to the model registry. We have prepared a script called `register_model.py`, which will check the results from the previous step and select the top 5 runs. After that, it will calculate the RMSE of those models on the test set (March 2022 data) and save the results to a new experiment called `random-forest-best-models`.

Your task is to update the script `register_model.py` so that it selects the model with the lowest RMSE on the test set and registers it to the model registry.

Tips for MLflow:

you can use the method search_runs from the MlflowClient to get the model with the lowest RMSE,
to register the model you can use the method mlflow.register_model and you will need to pass the right model_uri in the form of a string that looks like this: `"runs:/<RUN_ID>/model"`, and the name of the model (make sure to choose a good one!).
What is the test RMSE of the best model?

- 1.885
- 2.185
- 2.555
- 2.955

In [23]:
!python3 register_model.py

Traceback (most recent call last):
  File "/Users/personas/Dropbox/side_projects/github/mlops-zoomcamp-edu/02-experiment-tracking/homework/register_model.py", line 87, in <module>
    run_register_model()
  File "/Users/personas/anaconda3/envs/exp-tracking-env/lib/python3.9/site-packages/click/core.py", line 1130, in __call__
    return self.main(*args, **kwargs)
  File "/Users/personas/anaconda3/envs/exp-tracking-env/lib/python3.9/site-packages/click/core.py", line 1055, in main
    rv = self.invoke(ctx)
  File "/Users/personas/anaconda3/envs/exp-tracking-env/lib/python3.9/site-packages/click/core.py", line 1404, in invoke
    return ctx.invoke(self.callback, **ctx.params)
  File "/Users/personas/anaconda3/envs/exp-tracking-env/lib/python3.9/site-packages/click/core.py", line 760, in invoke
    return __callback(*args, **kwargs)
  File "/Users/personas/Dropbox/side_projects/github/mlops-zoomcamp-edu/02-experiment-tracking/homework/register_model.py", line 73, in run_register_model
   

# Q6. Model metadata
---

Now explore your best model in the model registry using UI. What information does the model registry contain about each model?

--> Version number <--
Source experiment
Model signature
All the above answers are correct
